# Recommendation System - Degree-Normalised Combined Embeddings

This notebook combines semantic (BGE-M3) and structural (Node2Vec) embeddings to build a
recommendation system for WikiCS articles using cosine similarity.

### Key features:
1. **Power-Law Normalisation**: The Node2Vec component is divided by $\log(\text{degree}+1)^{\alpha}$
   where $\alpha$ is estimated via maximum-likelihood fitting of the degree distribution.
   This penalises high-degree hub nodes that would otherwise dominate recommendations.
2. **Popularity Bias Removal**: The degree normalisation ensures popular nodes are not
   over-represented purely due to their connectivity.
3. **Exact Title Matching**: All test queries use canonical Wikipedia article titles
   (e.g. `Machine_learning`, not `Machine Learning`).
4. **Fuzzy Fallback**: If an exact title is not found, `utils.resolve_title()` finds
   the highest-scoring fuzzy match above a threshold.


In [1]:
import sys
import os
REPO_ROOT = r'C:\programming\github-repos\graph-ending'
utils_parent = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki')
sys.path.insert(0, utils_parent)

import pickle, os, json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

from utils import (
    load_graph_data,
    fuzzy_search,
    resolve_title,
    load_embeddings,
    build_normalized_embeddings,
)

DATA_PATH     = "../../data/data-embeddings.json"
BGE_M3_PATH  = "../../cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
N2V_PATH     = "../../cap-embeddings/node2vec/master_embeddings.parquet"
CACHE_DIR    = "../../cache/recommendation-1"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Libraries and utils loaded.")

Libraries and utils loaded.


## 1. Load Graph and Embeddings

In [2]:
nodes, id_to_title, title_to_id, G = load_graph_data(DATA_PATH)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

df_bge, df_n2v = load_embeddings(BGE_M3_PATH, N2V_PATH)
print(f"BGE-M3  : {len(df_bge)} nodes, dim={len(df_bge['embedding'].iloc[0])}")
print(f"Node2Vec: {len(df_n2v)} nodes, dim={len(df_n2v['embedding'].iloc[0])}")

Graph: 11701 nodes, 291039 edges
BGE-M3  : 11701 nodes, dim=1024
Node2Vec: 11701 nodes, dim=128


## 2. Power-Law Degree Distribution Fit

Estimate the power-law exponent $\alpha$ of the undirected degree distribution
using the `powerlaw` library.  The fitted $\alpha$ is then used to penalise
high-degree nodes in the Node2Vec component:

$$e_{\text{n2v}}^{*} = e_{\text{n2v}} / \log(d+1)^{\alpha}$$


In [3]:
import powerlaw

degrees = [G.degree(n) for n in G.nodes()]

fit = powerlaw.Fit(np.array(degrees, dtype=float), discrete=True, verbose=False)
alpha = fit.power_law.alpha
xmin  = fit.power_law.xmin

print(f"Power-law fit results:")
print(f"  alpha (exponent) = {alpha:.4f}")
print(f"  xmin             = {xmin}")
print(f"  sigma (std err)  = {fit.power_law.sigma:.4f}")

ALPHA_FILE = os.path.join(CACHE_DIR, "alpha.pkl")
with open(ALPHA_FILE, "wb") as f:
    pickle.dump({"alpha": alpha, "xmin": xmin, "degrees": degrees}, f)
print(f"\nAlpha saved to {ALPHA_FILE}")

Power-law fit results:
  alpha (exponent) = 2.9981
  xmin             = 115.0
  sigma (std err)  = 0.0491

Alpha saved to ../../cache/recommendation-1\alpha.pkl


## 3. Build Degree-Normalised Combined Embeddings

Each node's final embedding is the concatenation of:
- **BGE-M3** (1024-dim): semantic content, unchanged
- **Node2Vec** (128-dim): divided by $\log(\text{degree}+1)^{\alpha}$ (popularity penalty)


In [4]:
df_combined = build_normalized_embeddings(df_bge, df_n2v, G, alpha)
print(f"Combined embeddings: {len(df_combined)} nodes, dim={len(df_combined['embedding'].iloc[0])}")

EMB_FILE = os.path.join(CACHE_DIR, "combined_embeddings.pkl")
with open(EMB_FILE, "wb") as f:
    pickle.dump(df_combined, f)
print(f"Saved to {EMB_FILE}")

Combined embeddings: 11701 nodes, dim=1152
Saved to ../../cache/recommendation-1\combined_embeddings.pkl


## 4. Precompute Similarity Matrix

In [5]:
embedding_matrix = np.stack(df_combined["embedding"].values)
node_ids_arr = df_combined["id"].values
id_to_idx    = {nid: i for i, nid in enumerate(node_ids_arr)}
idx_to_id    = {i: nid for i, nid in enumerate(node_ids_arr)}
n_nodes      = len(node_ids_arr)

SIM_FILE = os.path.join(CACHE_DIR, "similarity_matrix.pkl")

if os.path.exists(SIM_FILE):
    print("  [cache] Loading similarity matrix...")
    with open(SIM_FILE, "rb") as f:
        sim_matrix = pickle.load(f)
    print(f"  [cache] Loaded: shape {sim_matrix.shape}")
else:
    print("  [comp]  Computing similarity matrix...")
    sim_matrix = cosine_similarity(embedding_matrix)
    with open(SIM_FILE, "wb") as f:
        pickle.dump(sim_matrix, f)
    print(f"  [save]  Saved.")

np.fill_diagonal(sim_matrix, 0.0)

existing_edges = set(G.edges())

def is_linked(src_id, tgt_id):
    return (src_id, tgt_id) in existing_edges

MAPPINGS_FILE = os.path.join(CACHE_DIR, "node_mappings.pkl")
with open(MAPPINGS_FILE, "wb") as f:
    pickle.dump({
        "node_ids_arr": node_ids_arr,
        "id_to_idx":    id_to_idx,
        "idx_to_id":    idx_to_id,
        "id_to_title":  id_to_title,
        "title_to_id":  title_to_id,
    }, f)

print(f"\nReady: {sim_matrix.shape}, {len(existing_edges)} edges")

  [cache] Loading similarity matrix...
  [cache] Loaded: shape (11701, 11701)

Ready: (11701, 11701), 291039 edges


## 5. Recommendation Functions

In [6]:
def get_recommendations(query, top_k=10, verbose=True):
    """
    Get top-k recommendations for a query using degree-normalised cosine similarity.
    1. Try exact title match in title_to_id.
    2. Fall back to resolve_title() (fuzzy, best score wins).
    Returns list of dicts: {title, similarity, linked}.
    """
    matched = resolve_title(query, title_to_id)
    if matched is None:
        if verbose:
            print(f"No match found for '{query}'.")
        return None
    if matched != query and verbose:
        print(f"(resolved '{query}' -> '{matched}')")

    node_id = title_to_id[matched]
    idx     = id_to_idx[node_id]
    sims    = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        rec_id = node_ids_arr[i]
        if rec_id == node_id:
            continue
        results.append({
            "title":      id_to_title[rec_id],
            "similarity": round(float(sims[i]), 6),
            "linked":     is_linked(node_id, rec_id),
        })

    return results[:top_k]


def display_recs(query, top_k=10):
    recs = get_recommendations(query, top_k=top_k)
    if recs is None:
        return
    print(f"\nTop-{top_k} for '{query}':")
    display(pd.DataFrame(recs))
    return recs


## 6. Exact-Match Test Queries

All queries below are exact Wikipedia article titles from the dataset.
They cover broad CS topics to test semantic understanding.

In [7]:
# Vague / broad topic queries (all exact Wikipedia article titles in the graph)
vague_queries = [
    "Natural_language_processing",
    "Operating_system",
    "Programming_language",
    "Database",
    "Cloud_computing",
    "World_Wide_Web",
    "Adversarial_machine_learning",
    "Brownout_(software_engineering)",
    "Computer_and_network_surveillance",
    "Quantum_programming",
]

print(f"Testing {len(vague_queries)} vague/broad topic queries...\n")
for q in vague_queries:
    recs = get_recommendations(q, top_k=5, verbose=False)
    if recs:
        tops = ", ".join([f"{r['title']} ({r['similarity']:.3f})" for r in recs])
        print(f"[VAGUE] {q}")
        print(f"       {tops}")
    else:
        print(f"[NO MATCH] {q}")
    print()


Testing 10 vague/broad topic queries...

[VAGUE] Natural_language_processing
       Semantic_parsing (0.728), Machine_translation (0.715), Natural_Language_Toolkit (0.704), Natural-language_generation (0.697), Language_model (0.696)

[VAGUE] Operating_system
       History_of_operating_systems (0.813), Disk_operating_system (0.783), Network_operating_system (0.772), Process_(computing) (0.767), Computer_multitasking (0.753)

[VAGUE] Programming_language
       Programming_language_generations (0.784), Winbatch (0.780), Syntax_(programming_languages) (0.776), Compiler (0.769), High-level_programming_language (0.767)

[VAGUE] Database
       Database_design (0.769), Federated_database_system (0.754), Database-centric_architecture (0.752), Outline_of_databases (0.749), Database_server (0.739)

[VAGUE] Cloud_computing
       Cloud_storage (0.783), Elastic_cloud_storage (0.769), Cloud_collaboration (0.758), Carrier_cloud (0.736), Software_as_a_service (0.734)

[VAGUE] World_Wide_Web
       

## 7. Detailed Recommendations

In [8]:
# Detailed recommendations for vague/broad topics
display_recs("Natural_language_processing")
display_recs("Operating_system")
display_recs("Programming_language")
display_recs("Database")
display_recs("Cloud_computing")
display_recs("World_Wide_Web")
print("\n---\n")


Top-10 for 'Natural_language_processing':


,title,similarity,linked
0,Semantic_parsing,0.727633,False
1,Machine_translation,0.714688,True
2,Natural_Language_Toolkit,0.704186,False
3,Natural-language_generation,0.696851,True
4,Language_model,0.695851,False
5,Computational_linguistics,0.689000,True
6,Computational_lexicology,0.682610,False
7,Question_answering,0.671226,True
8,Speech_recognition,0.669453,True
9,Cache_language_model,0.663527,True



Top-10 for 'Operating_system':


,title,similarity,linked
0,History_of_operating_systems,0.812929,True
1,Disk_operating_system,0.782788,True
2,Network_operating_system,0.772293,True
3,Process_(computing),0.767049,True
4,Computer_multitasking,0.752872,True
5,Multi-user_software,0.748936,True
6,System_virtual_machine,0.742693,False
7,Input/Output_Supervisor,0.739668,False
8,Real-time_operating_system,0.738556,True
9,Usage_share_of_operating_systems,0.735022,True



Top-10 for 'Programming_language':


...


Top-10 for 'Cloud_computing':


,title,similarity,linked
0,Cloud_storage,0.782910,True
1,Elastic_cloud_storage,0.768984,False
2,Cloud_collaboration,0.757899,True
3,Carrier_cloud,0.735901,True
4,Software_as_a_service,0.734464,True
5,Cloud_computing_issues,0.733118,True
6,Utility_computing,0.731475,True
7,Cloud_computing_architecture,0.726105,False
8,Cloud_testing,0.720434,False
9,SAP_Converged_Cloud,0.720299,False



Top-10 for 'World_Wide_Web':


,title,similarity,linked
0,Line_Mode_Browser,0.755768,False
1,Semantic_web_data_space,0.750845,False
2,Linked_data,0.745754,True
3,Web_server,0.716124,True
4,CERN_httpd,0.697279,False
5,Libwww,0.695723,False
6,Link_rot,0.692118,True
7,Web_interoperability,0.688781,False
8,URL,0.675170,True
9,Solid_(web_decentralization_project),0.672860,False



---



## 8. Batch Evaluation Table

In [9]:
def get_top_10_table(titles):
    rows = []
    for title in titles:
        recs = get_recommendations(title, top_k=10, verbose=False)
        if recs is None:
            print(f"Warning: '{title}' not found.")
            continue
        row = {"Query": title}
        for i, res in enumerate(recs):
            flag = " [LINK]" if res["linked"] else ""
            row[f"Top {i+1}"] = f"{res['title']} ({res['similarity']:.4f}){flag}"
        rows.append(row)
    return pd.DataFrame(rows)

sample_titles = list(title_to_id.keys())[:20]
display(get_top_10_table(sample_titles))

,Query,Top 1,Top 2,Top 3,Top 4,Top 5,Top 6,Top 7,Top 8,Top 9,Top 10
0,Twilio,Moxie_Marlinspike (0.6779),Signal_Protocol (0.6757),FireEye (0.6672),Twister_(software) (0.6516),Cloudflare (0.6462),Nginx (0.6432),Mozilla_Thunderbird (0.6432),Typeform_(service) (0.6381),Zoho_Corporation (0.6358),"Workday,_Inc. (0.6343)"
1,Program_compatibility_date_range,Processor_Control_Region (0.7545),XOFTspy_Portable_Anti-Spyware (0.7420),ZAP_File (0.7374),Auslogics_Antivirus (0.7289),Spam_Reader (0.7284),SUSE_Studio_ImageWriter (0.7259),StegAlyzerAS (0.7250),Reboot_Restore_Rx (0.7220),RunScanner (0.7219),GDocsDrive (0.7213)
2,SYSTAT_(DEC),TOPS-10 (0.9809) [LINK],TOPS-20 (0.7768),PDP-10 (0.7213),RSTS/E (0.6829) [LINK],MS-DOS (0.6721),TENEX_(operating_system) (0.6703),IBM_TopView (0.6639),Incompatible_Timesharing_System (0.6632),Universal_Time-Sharing_System (0.6504),DEC_BATCH-11/DOS-11 (0.6501)
3,List_of_column-oriented_DBMSes,List_of_relational_database_management_systems...,List_of_cluster_management_software (0.6726),Column-oriented_DBMS (0.6665) [LINK],Epictetus_Database_Client (0.6495),Comparison_of_object-relational_database_manag...,Database_administration (0.6431),Oracle_Exadata (0.6398) [LINK],List_of_in-memory_databases (0.6351),Embedded_database (0.6274),Virtual_column (0.6271)
4,Stealth_wallpaper,Wireless_intrusion_prevention_system (0.5668),Warchalking (0.5660),Air_gap_(networking) (0.5629),Secure_transmission (0.5597),Air-gap_malware (0.5511),Stegomalware (0.5488),Wireless_security (0.5441) [LINK],Black_hole_(networking) (0.5439),Cracking_of_wireless_networks (0.5412),IP_address_spoofing (0.5387)
5,Scalable_TCP,HSTCP (0.7370),TCP_window_scale_option (0.7136),TCP_tuning (0.6985),TCP-Illinois (0.6982),TCP_global_synchronization (0.6953),Compound_TCP (0.6946),Zeta-TCP (0.6889),BIC_TCP (0.6887),FAST_TCP (0.6830),CUBIC_TCP (0.6776)
6,Carrier_IQ,Microsoft_SmartScreen (0.6619),Webroot (0.6581),Spy_Sweeper (0.6554),Webroot_Antivirus_with_Spy_Sweeper (0.6551),Webroot_Internet_Security_Complete (0.6549),QUIC (0.6535),Webroot_Internet_Security_Essentials (0.6490),Hewlett-Packard (0.6459),IMessage (0.6432),HP_Autonomy (0.6420)
7,ACF2,Resource_Access_Control_Facility (0.7229) [LINK],VM_(operating_system) (0.6268) [LINK],IBM_Airline_Control_Program (0.6166),Secure_Computing_Corporation (0.6108),IBM_System_Management_Facilities (0.6084),OpenAFS (0.6044),Andrew_File_System (0.6036),Virtual_Telecommunications_Access_Method (0.59...,ICL_VME (0.5929),Automated_Certificate_Management_Environment (...
8,Dorkbot_(malware),Code_Shikara (0.8256),Slenfbot (0.7391),Anti-worm (0.6921),Browser_hijacking (0.6740),Greynet (0.6728),Torpig (0.6711),Sality (0.6674),Internet_security (0.6573),Storm_botnet (0.6551),Duqu (0.6550)
9,Lout_(software),Windows_RSS_Platform (0.9984),MetaSAN (0.9983),EFx_Factory (0.9983),RScheme (0.9983),Specfs (0.9983),CLX_(Common_Lisp) (0.9983),Weblocks (0.9983),ScriptBasic (0.9983),Comparison_of_database_access (0.9983),Remote_Application_Programming_Interface (0.9983)
